# Week 68, Day 4: Rust extensions with PyO3

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w068d4.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
rust_to_python = {"i64": "int", "f64": "float", "bool": "bool", "String": "str", "&str": "str"}

def python_signature(name, params, returns):
    args = ", ".join(f"{p}: {rust_to_python[t]}" for p, t in params)
    return f"def {name}({args}) -> {rust_to_python[returns]}: ..."

print(python_signature("area", [("w", "f64"), ("h", "f64")], "f64"))
print(python_signature("shout", [("text", "&str")], "String"))

In [ ]:
import re

rust = '''
#[pyfunction]
fn mean(values: Vec<f64>) -> PyResult<f64> { ... }

fn helper(x: f64) -> f64 { x * 2.0 }

#[pyfunction]
fn shout(text: &str) -> String { text.to_uppercase() }
'''
exported = re.findall(r"#\[pyfunction\]\s*fn (\w+)", rust)
everything = re.findall(r"fn (\w+)", rust)
print("visible from Python:", exported)
print("Rust-only:", sorted(set(everything) - set(exported)))

In [ ]:
SIMPLE = {"i32": "int", "i64": "int", "u64": "int", "usize": "int", "f64": "float",
          "bool": "bool", "String": "str", "&str": "str"}

def python_type(rust):
    rust = rust.strip()
    for wrapper, template in (("Vec<", "list[{}]"), ("Option<", "{} | None"), ("PyResult<", "{}")):
        if rust.startswith(wrapper) and rust.endswith(">"):
            return template.format(python_type(rust[len(wrapper):-1]))
    return SIMPLE.get(rust, "object")

for t in ["f64", "Vec<f64>", "Option<String>", "PyResult<Vec<i64>>", "PyResult<Option<&str>>"]:
    print(f"{t:24} -> {python_type(t)}")

In [ ]:
class Ok:
    def __init__(self, value):
        self.value = value
    def __repr__(self):
        return f"Ok({self.value!r})"

class Err:
    def __init__(self, error):
        self.error = error
    def __repr__(self):
        return f"Err({self.error!r})"

def parse_int(text):
    return Ok(int(text)) if text.strip().lstrip("-").isdigit() else Err(f"invalid digit in {text!r}")

def add_parsed(a, b):
    "What Rust's ? does: return the first error, otherwise carry on with the values."
    x = parse_int(a)
    if isinstance(x, Err):
        return x
    y = parse_int(b)
    if isinstance(y, Err):
        return y
    return Ok(x.value + y.value)

print(add_parsed("2", "40"), add_parsed("2", "forty"))

In [ ]:
def check(ops):
    "A tiny borrow checker: ops are (action, variable)."
    moved, shared, mutable, errors = set(), {}, set(), []
    for action, var in ops:
        if action in ("use", "move", "borrow", "borrow_mut") and var in moved:
            errors.append(f"use of moved value {var}")
            continue
        if action == "move":
            moved.add(var)
        elif action == "borrow":
            if var in mutable:
                errors.append(f"{var} is already borrowed as mutable")
            shared[var] = shared.get(var, 0) + 1
        elif action == "borrow_mut":
            if var in mutable or shared.get(var):
                errors.append(f"{var} is already borrowed")
            mutable.add(var)
        elif action == "release":
            shared.pop(var, None)
            mutable.discard(var)
    return errors

print(check([("borrow", "v"), ("borrow", "v"), ("release", "v"), ("borrow_mut", "v")]))
print(check([("move", "s"), ("use", "s")]))
print(check([("borrow", "v"), ("borrow_mut", "v")]))

In [ ]:
def wheel_name(project, version, python_tag, abi_tag, platform):
    return f"{project}-{version}-{python_tag}-{abi_tag}-{platform}.whl"

print(wheel_name("stats", "0.1.0", "cp39", "abi3", "manylinux_2_17_x86_64"))
print(wheel_name("stats", "0.1.0", "cp313", "cp313", "macosx_11_0_arm64"))
print("the first works on any CPython from 3.9 on Linux x86-64; the second only on CPython 3.13 on Apple silicon")

> **Key idea.** PyO3 connects Rust to Python: #[pyfunction] exposes functions (doc comments become docstrings), #[pyclass] structs become classes with #[pyo3(get, set)] fields, #[pymethods] exposes methods with #[new] as the constructor, and #[pymodule] assembles the module. Arguments and results are converted automatically (integers with range checks, f64, bool, String and &str, Vec, HashMap, Option as None, tuples), and .pyi stub files describe compiled modules to type checkers. Rust reports failure with Result (Ok or Err) and absence with Option (Some or None); ? returns early on errors; PyResult errors such as PyValueError::new_err become Python exceptions, and panics become PanicException. Ownership (one owner, moves, borrows: many shared or one mutable) is checked at compile time, preventing use-after-free and data races. maturin new, develop and build make, install and package the extension, with abi3 for one wheel across versions.

### Exercise: Rust types as Python types

Write `python_type(rust)`: the Python type hint for a Rust type, as a string: integers (`i8` to `i64`, `u8` to `u64`, `isize`, `usize`) give `int`; `f32` and `f64` give `float`; `bool` gives `bool`; `String` and `&str` give `str`; `Vec<T>` gives `list[T]`; `HashMap<K, V>` gives `dict[K, V]`; `Option<T>` gives `T | None`; `PyResult<T>` gives `T`; a tuple `(A, B, ...)` gives `tuple[A, B, ...]`; `()` gives `None`; anything else gives `object`. Nested types must work, so split at the **top-level** commas only.

In [ ]:
INTS = {"i8", "i16", "i32", "i64", "u8", "u16", "u32", "u64", "isize", "usize"}

def python_type(rust):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_types():\n    \"type conversion\"\n    cases = {\n        \"i64\": \"int\", \"usize\": \"int\", \"f32\": \"float\", \"bool\": \"bool\", \"&str\": \"str\", \"String\": \"str\",\n        \"Vec<f64>\": \"list[float]\", \"Option<String>\": \"str | None\", \"PyResult<Vec<i64>>\": \"list[int]\",\n        \"HashMap<String, Vec<u8>>\": \"dict[str, list[int]]\", \"(i32, f64)\": \"tuple[int, float]\",\n        \"()\": \"None\", \"PyResult<()>\": \"None\", \"Vec<(String, Option<i64>)>\": \"list[tuple[str, int | None]]\",\n        \"Counter\": \"object\",\n    }\n    for rust, py in cases.items():\n        assert python_type(rust) == py, rust")

### Exercise: Stubs for a compiled module

Write `stub_lines(rust_source)`: for every function marked `#[pyfunction]` (the attribute on its own line just before `fn`), produce a stub line `def name(param: type, ...) -> type: ...` using a `python_type` function (given in the starter). Parameters named `py` (PyO3's GIL token, `py: Python<'_>`) are left out; a function with no `->` returns `None`. Functions without the attribute are skipped. If the function has a `///` doc comment directly above the attribute, add it as the docstring: `def mean(values: list[float]) -> float:` followed by an indented line `"""Average of a list."""` and `    ...`, all as separate list items.

In [ ]:
import re

def python_type(rust):
    simple = {"i32": "int", "i64": "int", "u64": "int", "usize": "int", "f64": "float",
              "bool": "bool", "String": "str", "&str": "str", "()": "None"}
    rust = rust.strip()
    for wrapper, template in (("Vec<", "list[{}]"), ("Option<", "{} | None"), ("PyResult<", "{}")):
        if rust.startswith(wrapper) and rust.endswith(">"):
            return template.format(python_type(rust[len(wrapper):-1]))
    return simple.get(rust, "object")

def stub_lines(rust_source):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_stubs():\n    \"generating .pyi stubs\"\n    src = '''\n/// Average of a list.\n#[pyfunction]\nfn mean(values: Vec<f64>) -> PyResult<f64> {\n    Ok(values.iter().sum::<f64>() / values.len() as f64)\n}\n\nfn helper(x: f64) -> f64 { x }\n\n#[pyfunction]\nfn shout(py: Python<'_>, text: &str, times: usize) -> String {\n    text.repeat(times)\n}\n\n#[pyfunction]\nfn reset() {\n}\n'''\n    assert stub_lines(src) == [\n        \"def mean(values: list[float]) -> float:\",\n        '    \"\"\"Average of a list.\"\"\"',\n        \"    ...\",\n        \"def shout(text: str, times: int) -> str: ...\",\n        \"def reset() -> None: ...\",\n    ]\n    assert stub_lines(\"fn private() {}\") == []")

### Exercise: Result and the ? operator

Write Python versions of Rust's `Result`: classes `Ok(value)` and `Err(error)`, both with `is_ok()`, `unwrap()` (the value, or for `Err` raise `RuntimeError(f"called unwrap on Err: {error}")`, like a panic), `unwrap_or(default)`, `map(func)` (applies `func` to an `Ok`'s value, leaving an `Err` unchanged), `and_then(func)` (for `Ok`, returns `func(value)`, which is itself a result) and `==`. Then `collect(results)`: like Rust's `?` in a loop: `Ok(list of values)` if every result is `Ok`, otherwise the **first** `Err`.

In [ ]:
class Ok:
    def __init__(self, value):
        ___

class Err:
    def __init__(self, error):
        ___

def collect(results):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_result():\n    \"Result\"\n    assert Ok(2).map(lambda x: x * 10) == Ok(20) and Err(\"bad\").map(lambda x: x * 10) == Err(\"bad\")\n    assert Ok(5).unwrap() == 5 and Err(\"e\").unwrap_or(0) == 0 and Ok(1).is_ok() and not Err(1).is_ok()\n    half = lambda x: Ok(x // 2) if x % 2 == 0 else Err(f\"{x} is odd\")\n    assert Ok(8).and_then(half).and_then(half) == Ok(2) and Ok(6).and_then(half).and_then(half) == Err(\"3 is odd\")\n    try:\n        Err(\"disk full\").unwrap()\n        assert False\n    except RuntimeError as e:\n        assert str(e) == \"called unwrap on Err: disk full\"\n    assert collect([Ok(1), Ok(2)]) == Ok([1, 2]) and collect([Ok(1), Err(\"a\"), Err(\"b\")]) == Err(\"a\")\n    assert collect([]) == Ok([]) and Ok(1) != Err(1)")

### Exercise: A tiny borrow checker

Simulate Rust's ownership rules. Operations are `(action, variable)` pairs: `"let"` (create an owned value), `"move"` (the value moves away to another owner), `"copy"` (allowed any time: it's a copyable type), `"use"`, `"borrow"` (a shared reference), `"borrow_mut"` (a mutable reference) and `"end"` (all borrows of that variable end). Write `borrow_errors(ops)`: a list of error messages, in order: `"use of moved value: x"` for any use, move or borrow after a move; `"cannot borrow x as mutable because it is also borrowed as immutable"`; `"cannot borrow x as mutable more than once at a time"`; `"cannot borrow x as immutable because it is also borrowed as mutable"`; and `"x is not declared"` for using a variable never created with `let`. An operation that causes an error doesn't change the state.

In [ ]:
def borrow_errors(ops):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_borrow():\n    \"ownership and borrowing\"\n    assert borrow_errors([(\"let\", \"v\"), (\"borrow\", \"v\"), (\"borrow\", \"v\"), (\"end\", \"v\"), (\"borrow_mut\", \"v\"), (\"use\", \"v\")]) == []\n    assert borrow_errors([(\"let\", \"s\"), (\"move\", \"s\"), (\"use\", \"s\")]) == [\"use of moved value: s\"]\n    assert borrow_errors([(\"let\", \"v\"), (\"borrow\", \"v\"), (\"borrow_mut\", \"v\")]) == [\n        \"cannot borrow v as mutable because it is also borrowed as immutable\"]\n    assert borrow_errors([(\"let\", \"v\"), (\"borrow_mut\", \"v\"), (\"borrow_mut\", \"v\"), (\"borrow\", \"v\")]) == [\n        \"cannot borrow v as mutable more than once at a time\",\n        \"cannot borrow v as immutable because it is also borrowed as mutable\"]\n    assert borrow_errors([(\"use\", \"ghost\")]) == [\"ghost is not declared\"]\n    assert borrow_errors([(\"let\", \"n\"), (\"copy\", \"n\"), (\"copy\", \"n\"), (\"use\", \"n\")]) == []\n    assert borrow_errors([(\"let\", \"a\"), (\"move\", \"a\"), (\"borrow\", \"a\"), (\"let\", \"a\"), (\"use\", \"a\")]) == [\"use of moved value: a\"]")

### Exercise: Rust errors as Python exceptions

PyO3 raises a Python exception for each Rust error type it's told about. Write `raise_for(err)`: given a Rust-style error as a `(kind, message)` pair, raise the matching Python exception from `ERRORS` (`"PyValueError"` to `ValueError`, and so on), and for an unknown kind raise `RuntimeError` with the message prefixed by the kind (`"IoError: disk full"`). Then `call_rust(func, *args)`: call a function returning an `("ok", value)` or `("err", (kind, message))` pair, return the value or raise as `raise_for` does, and turn a Python exception raised **inside** `func` (a "panic") into `RuntimeError("panic: " + str(e))`.

In [ ]:
ERRORS = {"PyValueError": ValueError, "PyTypeError": TypeError, "PyKeyError": KeyError,
          "PyIndexError": IndexError, "PyZeroDivisionError": ZeroDivisionError}

def raise_for(err):
    ___

def call_rust(func, *args):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_errors():\n    \"exceptions across the boundary\"\n    def mean(values):\n        if not values:\n            return (\"err\", (\"PyValueError\", \"mean of empty list\"))\n        return (\"ok\", sum(values) / len(values))\n    assert call_rust(mean, [1, 2, 3]) == 2.0\n    try:\n        call_rust(mean, [])\n        assert False\n    except ValueError as e:\n        assert str(e) == \"mean of empty list\"\n    try:\n        raise_for((\"IoError\", \"disk full\"))\n        assert False\n    except RuntimeError as e:\n        assert str(e) == \"IoError: disk full\"\n    def panics(xs):\n        return (\"ok\", xs[10])\n    try:\n        call_rust(panics, [1])\n        assert False\n    except RuntimeError as e:\n        assert str(e) == \"panic: list index out of range\"")

### Bug hunt: Unwrapping a None

The lookup returns a Rust-style `Option` (here `None` when the key is missing), and calling the simulated `unwrap` on it panics, so this crashes with `RuntimeError`. Use a default for a missing value instead (Rust's `unwrap_or`), so this prints `0`.

In [ ]:
def unwrap(option):
    if option is None:
        raise RuntimeError("called `Option::unwrap()` on a `None` value")
    return option

def unwrap_or(option, default):
    return default if option is None else option

stock = {"tea": 5}
print(unwrap(stock.get("jam")))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"handle None\"\n    expect_output(\"0\")")

### Bug hunt: A stub that told the wrong story

The Rust function returns `PyResult<Vec<f64>>`, but the stub generator only strips `PyResult` and maps the rest as a plain name, so the stub says `-> object` and this prints `def scores() -> object: ...`. Map the inner `Vec<f64>` too (recursively) so it prints `def scores() -> list[float]: ...`.

In [ ]:
SIMPLE = {"f64": "float", "i64": "int", "String": "str"}

def python_type(rust):
    if rust.startswith("PyResult<"):
        rust = rust[len("PyResult<"):-1]
    return SIMPLE.get(rust, "object")

print(f"def scores() -> {python_type('PyResult<Vec<f64>>')}: ...")

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"recursive types\"\n    expect_output(\"def scores() -> list[float]: ...\")")

### Bug hunt: An error that wasn't passed on

This imitation of Rust's `?` checks the first result but ignores an `Err` in the second, so it adds a string to a number and crashes with `TypeError`. Return the second error too, as `?` would, so this prints `Err: bad digit 'x'`.

In [ ]:
def parse(text):
    return ("ok", int(text)) if text.isdigit() else ("err", f"bad digit {text!r}")

def add(a, b):
    x = parse(a)
    if x[0] == "err":
        return x
    y = parse(b)
    return ("ok", x[1] + y[1])

result = add("2", "x")
print(f"Err: {result[1]}" if result[0] == "err" else result[1])

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"pass errors on\"\n    expect_output(\"Err: bad digit 'x'\")")

### Puzzle: A PyO3 function

Arrange the lines of a Rust function exposed to Python that raises `ValueError` for an empty list (as you'd write it in `src/lib.rs`). One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # the Python type for the Rust type Option<i64>, as a string
q = ___   # the attribute that exposes a Rust function to Python
r = ___   # True or False: in Rust, a value can have a mutable borrow and a shared borrow at the same time
s = ___   # the Python exception for Err(PyKeyError::new_err("x")), as a string
t = ___   # the maturin command that compiles and installs into the current environment

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == \"int | None\" and q == \"#[pyfunction]\" and r is False, \"p, q and r\"\n    assert s == \"KeyError\" and t == \"maturin develop\", \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: a stub file for a whole module

Write `module_stub(rust_source)`: the text of a `.pyi` stub for a PyO3 module, built from:

- `#[pyfunction]` functions, as in the stubs exercise (without docstrings), excluding a `py` parameter;
- `#[pyclass] struct Name { ... }` blocks: a `class Name:` line, then, for each field with `#[pyo3(get)]` or `#[pyo3(get, set)]` on the line before it, an indented `field: type` line (fields without those aren't visible from Python);
- `#[pymethods] impl Name { ... }` blocks: inside the class, a method stub for each `fn`: the constructor (marked `#[new]`) becomes `def __init__(self, ...) -> None: ...`; methods with `&self` or `&mut self` become `def name(self, ...) -> type: ...`; the `self` parameter is written as `self`.

Use the `python_type` helper in the starter. Output, in order: the function stubs, then each class (fields first, then methods), with a blank line before each class. Classes with nothing visible get an indented `...`.

In [ ]:
import re

def python_type(rust):
    simple = {"i32": "int", "i64": "int", "u64": "int", "usize": "int", "f64": "float",
              "bool": "bool", "String": "str", "&str": "str", "()": "None", "Self": "None"}
    rust = rust.strip()
    for wrapper, template in (("Vec<", "list[{}]"), ("Option<", "{} | None"), ("PyResult<", "{}")):
        if rust.startswith(wrapper) and rust.endswith(">"):
            return template.format(python_type(rust[len(wrapper):-1]))
    return simple.get(rust, "object")

def module_stub(rust_source):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_module_stub():\n    \"a .pyi for a module\"\n    src = '''\n#[pyfunction]\nfn mean(values: Vec<f64>) -> PyResult<f64> {\n    Ok(0.0)\n}\n\n#[pyclass]\nstruct Counter {\n    #[pyo3(get)]\n    count: u64,\n    step: u64,\n    #[pyo3(get, set)]\n    label: String,\n}\n\n#[pymethods]\nimpl Counter {\n    #[new]\n    fn new(step: u64) -> Self {\n        Counter { count: 0, step, label: String::new() }\n    }\n    fn add(&mut self, times: u64) {\n        self.count += self.step * times;\n    }\n    fn describe(&self) -> String {\n        format!(\"{}\", self.count)\n    }\n}\n\n#[pyclass]\nstruct Empty {\n    hidden: i64,\n}\n'''\n    assert module_stub(src) == (\n        \"def mean(values: list[float]) -> float: ...\\n\"\n        \"\\n\"\n        \"class Counter:\\n\"\n        \"    count: int\\n\"\n        \"    label: str\\n\"\n        \"    def __init__(self, step: int) -> None: ...\\n\"\n        \"    def add(self, times: int) -> None: ...\\n\"\n        \"    def describe(self) -> str: ...\\n\"\n        \"\\n\"\n        \"class Empty:\\n\"\n        \"    ...\"\n    )")

## Warm-up

A Rust function's signature says exactly what types it takes and returns, and PyO3 converts between those and Python's types automatically. Writing out the Python view of each function (a **stub**) is how type checkers and editors learn what a compiled module contains.

## A PyO3 module

Here is a complete Rust extension module (in `src/lib.rs`), with a function and a class:

```
use pyo3::prelude::*;
use pyo3::exceptions::PyValueError;

/// Average of a list of numbers.
#[pyfunction]
fn mean(values: Vec<f64>) -> PyResult<f64> {
    if values.is_empty() {
        return Err(PyValueError::new_err("mean of empty list"));
    }
    Ok(values.iter().sum::<f64>() / values.len() as f64)
}

#[pyclass]
struct Counter {
    #[pyo3(get)]
    count: u64,
}

#[pymethods]
impl Counter {
    #[new]
    fn new() -> Self {
        Counter { count: 0 }
    }
    fn add(&mut self, n: u64) {
        self.count += n;
    }
}

#[pymodule]
fn stats(m: &Bound<'_, PyModule>) -> PyResult<()> {
    m.add_function(wrap_pyfunction!(mean, m)?)?;
    m.add_class::<Counter>()?;
    Ok(())
}
```

In Python: `import stats; stats.mean([1, 2, 3])` gives `2.0`; `stats.mean([])` raises `ValueError`; `c = stats.Counter(); c.add(5); c.count` gives `5`, and `c.count = 9` fails because only `get` was allowed. The parts:

- `#[pyfunction]` makes a Rust function callable from Python; its doc comment (`///`) becomes `__doc__`.
- `#[pyclass]` makes a Rust struct into a Python class; `#[pyo3(get, set)]` exposes fields as attributes.
- `#[pymethods]` exposes methods; `#[new]` is the constructor; `&self` and `&mut self` methods read or change the object.
- `#[pymodule]` defines the module and what it contains.

## Types across the boundary

PyO3 converts arguments from Python to Rust when a function is called, and results back when it returns. The main conversions:

| Rust | Python |
|---|---|
| `i8` ... `i64`, `u8` ... `u64`, `isize`, `usize` | `int` (out-of-range values raise `OverflowError`) |
| `f32`, `f64` | `float` |
| `bool` | `bool` |
| `String`, `&str` | `str` |
| `Vec<T>` | `list` (any sequence is accepted as an argument) |
| `HashMap<K, V>` | `dict` |
| `Option<T>` | `T` or `None` |
| `(A, B)` | `tuple` |
| `PyResult<T>` | `T`, or an exception |

Converting a `Vec<f64>` copies the whole list into Rust memory, which costs time for big inputs; PyO3 can also borrow Python objects directly (`&Bound<PyList>`), and the `numpy` crate gives zero-copy access to NumPy arrays (the buffer protocol, yesterday).

Because a compiled module has no Python source, type checkers and editors can't see its signatures. Projects ship a **stub file**, `stats.pyi`, with the Python signatures and `...` bodies (PEP 484), so tools know that `mean` takes `list[float]` and returns `float`.

## Errors: Result, Option and exceptions

Rust has no exceptions. A function that can fail returns a **`Result<T, E>`**: either `Ok(value)` or `Err(error)`. A value that may be missing is an **`Option<T>`**: `Some(value)` or `None`. The compiler insists that you handle both cases before using the value, which is why Rust code rarely crashes from an unexpected `None`.

- `match result { Ok(v) => ..., Err(e) => ... }` handles both cases explicitly.
- The **`?` operator** after an expression returns early with the error if it's an `Err`, and otherwise unwraps the `Ok` value: `let n: i64 = text.parse()?;`.
- `.unwrap()` takes the value and **panics** (crashes) if there isn't one; PyO3 turns a panic into a `PanicException` in Python rather than crashing the interpreter, but it's still a bug.

At the boundary, **`PyResult<T>`** is `Result<T, PyErr>`. Returning `Err(PyValueError::new_err("..."))` raises `ValueError` in Python; there are matching types for `PyTypeError`, `PyKeyError`, `PyIndexError` and so on. Rust error types can be converted to Python exceptions automatically by implementing `From<MyError> for PyErr`, which lets `?` turn a Rust error straight into a Python exception.

## Ownership and borrowing

Rust's safety comes from rules the compiler checks before the program ever runs:

- **Every value has exactly one owner**, a variable. When the owner goes out of scope, the value is freed (no garbage collector, no reference counts for ordinary values).
- **Assigning or passing a value moves it**: `let b = a;` makes `b` the owner, and using `a` afterwards is a compile-time error ("use of moved value"). Types such as integers are simply **copied** instead.
- **Borrowing** lends access without moving: `&a` is a shared (read-only) reference, `&mut a` an exclusive (writable) one. At any moment a value may have **either** any number of shared borrows **or** exactly one mutable borrow, never both. And a borrow can't outlive the value it borrows (**lifetimes**).

Those rules rule out whole classes of bugs at compile time: use after free, double free, dangling pointers, iterator invalidation, and **data races** (two threads writing the same data needs two mutable borrows, which the rules forbid). That's why Rust is attractive for extensions: you get C's speed without Monday's reference-counting worries, and PyO3 can release the GIL (`py.allow_threads(|| ...)`) for parallel Rust code with the compiler checking it's safe.

## Building with maturin

**maturin** (a free tool, `pip install maturin`) builds and packages PyO3 projects. On your own computer, with Rust installed (`rustup` from rust-lang.org):

```
pip install maturin
maturin new stats            # choose "pyo3": creates Cargo.toml, pyproject.toml and src/lib.rs
cd stats
maturin develop              # compile and install into the current virtual environment
python -c "import stats; print(stats.mean([1, 2, 3]))"
maturin build --release      # build a wheel to distribute
```

`Cargo.toml` lists the Rust dependencies (`pyo3`, with the `abi3-py39` feature for one wheel that works on Python 3.9 and later, like Monday's stable ABI), and `pyproject.toml` tells `pip` to use maturin as the build backend. Projects usually publish pre-built wheels for each platform (with the free `maturin-action` in continuous integration), so users don't need Rust at all.

## Practice

## Quiz

## Daily challenge

## Before you go

On your own computer, install Rust with `rustup`, then follow the maturin steps above to build the `stats` module and call it from Python. Tomorrow: the most important question of the week, which is when extending Python is worth it at all.